# 09 - Refinamiento fino de los tres mejores candidatos

Este nodo no repite la búsqueda amplia. Espera a que el nodo 07 complete su evaluación OOF de cinco folds y selecciona exactamente tres puntos de partida:

1. los dos mejores candidatos por **log loss OOF cross-calibrado de cinco folds**;
2. el mejor candidato 3D;
3. si el 3D ya pertenece al top 2, completa el tercer cupo con el siguiente candidato del ranking.

Cada rama conserva arquitectura, pooling, variante y lateralidad. Optuna sólo explora un vecindario local informado por las mejores trayectorias del nodo 07: learning rate más bajo, regularización, dropout, intensidad de augmentation y configuración tabular. Cada trial usa tres folds; los tres ganadores se reentrenan después con cinco folds congelados.


In [1]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modeling.cnn.io import RunLock
from modeling.node09_refinement import (
    RefinementExperiment,
    prepare_refinement,
    run_final_stage,
    run_search_stage,
)

RUN_ID = os.environ.get("DAT_NODE09_RUN_ID", "node09_fine_v1")
SOURCE_NODE07_RUN_ID = os.environ.get("DAT_NODE09_SOURCE_RUN_ID", "dat_spect_slab_v4")
NODE4_PROFILE = os.environ.get("DAT_NODE09_NODE4_PROFILE", "v3")
TRIALS_PER_MODEL = int(os.environ.get("DAT_NODE09_TRIALS_PER_MODEL", "18"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENT = RefinementExperiment(
    run_id=RUN_ID,
    source_node07_run_id=SOURCE_NODE07_RUN_ID,
)
EXPERIMENT = replace(
    EXPERIMENT,
    data=replace(EXPERIMENT.data, node4_profile=NODE4_PROFILE),
    search=replace(EXPERIMENT.search, trials_per_model=TRIALS_PER_MODEL),
)
RUN_DIR = PROJECT_ROOT / "outputs" / "private_eda" / "node09_runs" / RUN_ID

display(Markdown(
    f"**Run nodo 09:** `{RUN_ID}` · **fuente nodo 07:** `{SOURCE_NODE07_RUN_ID}` · "
    f"**device:** `{DEVICE}` · **trials completos por rama:** `{TRIALS_PER_MODEL}` · "
    f"**máximo:** `{EXPERIMENT.search.max_epochs_search}` épocas"
))


**Run nodo 09:** `node09_fine_v1` · **fuente nodo 07:** `dat_spect_slab_v4` · **device:** `cuda` · **trials completos por rama:** `18` · **máximo:** `32` épocas

## 1. Gate de comparabilidad

La selección no se habilita con OOF parciales ni con el score de los tres folds de Optuna. Debe existir `final_metrics.csv` del nodo 07, lo que implica que sus tres finalistas ya pasaron por el mismo CV5.


In [2]:
SOURCE_RUN_DIR = (
    PROJECT_ROOT / "outputs" / "private_eda" / "node07_runs" / SOURCE_NODE07_RUN_ID
)
source_metrics_path = SOURCE_RUN_DIR / "final" / "final_metrics.csv"
source_finalists_path = SOURCE_RUN_DIR / "search" / "finalists.json"
source_oof = sorted((SOURCE_RUN_DIR / "final").glob("oof_*.csv"))
display(pd.DataFrame({
    "artefacto": ["finalistas búsqueda", "OOF finalistas completos", "ranking CV5 final"],
    "estado": [
        source_finalists_path.exists(),
        len(source_oof),
        source_metrics_path.exists(),
    ],
}))
if not source_metrics_path.exists():
    raise RuntimeError(
        "El nodo 07 todavía no terminó los tres finalistas CV5. "
        "Reanuda su celda final y vuelve aquí cuando exista final_metrics.csv."
    )
display(pd.read_csv(source_metrics_path))


,artefacto,estado
0,finalistas búsqueda,True
1,OOF finalistas completos,3
2,ranking CV5 final,True


,candidate_id,architecture,feature_variant,lateral_strategy,raw_log_loss,raw_brier,raw_auc,raw_balanced_accuracy_0_5,raw_ece_10,calibrated_log_loss,calibrated_brier,calibrated_auc,calibrated_balanced_accuracy_0_5,calibrated_ece_10,deployment_temperature
0,23f042f62f40d03e,slab2d,image_radiomics_sbr,random_flip,0.556924,0.187166,0.799906,0.735958,0.064860,0.551498,0.184972,0.796822,0.735958,0.045810,1.266840
1,fb93ea204b0ec32f,2.5d,image_radiomics_sbr,random_flip,0.560599,0.187826,0.797649,0.739593,0.071564,0.552242,0.184982,0.793100,0.739593,0.043068,1.333065
2,3f9bbf24164bd1dc,3d,image_radiomics_sbr,random_flip,0.586020,0.195744,0.780836,0.727110,0.087526,0.565622,0.190079,0.775836,0.727110,0.029703,1.508521


## 2. Preparación y selección de las tres referencias

Se reutilizan los mismos 1362 pacientes, caches y folds congelados. Esta etapa no vuelve a ejecutar registro ni extracción de características. La tabla permite comprobar qué modelo entró por top global, cuál por mejor 3D y cuál completó diversidad si hubo solapamiento.


In [3]:
with RunLock(RUN_DIR / "prepare.lock"):
    prepared = prepare_refinement(EXPERIMENT, project_root=PROJECT_ROOT)

references = pd.read_csv(RUN_DIR / "config" / "selected_references.csv")
display(references)
display(pd.DataFrame({
    "indicador": [
        "Pacientes únicos", "Ramas Optuna", "Folds por trial",
        "Trials completos por rama", "Entrenamientos CV de búsqueda objetivo",
        "Folds de evaluación final",
    ],
    "valor": [
        prepared.cohort["uid"].nunique(), 3,
        EXPERIMENT.search.n_splits_search,
        EXPERIMENT.search.trials_per_model,
        3 * EXPERIMENT.search.trials_per_model * EXPERIMENT.search.n_splits_search,
        EXPERIMENT.search.n_splits_final,
    ],
}))


,source_candidate_id,node07_rank,node07_calibrated_log_loss,selection_roles,architecture,feature_variant,lateral_strategy,source_learning_rate,source_fixed_epochs,source_completed_trials
0,23f042f62f40d03e,1,0.551498,top_1_overall,slab2d,image_radiomics_sbr,random_flip,0.000317,9,4
1,fb93ea204b0ec32f,2,0.552242,top_2_overall,2.5d,image_radiomics_sbr,random_flip,0.000317,9,2
2,3f9bbf24164bd1dc,3,0.565622,best_3d,3d,image_radiomics_sbr,random_flip,0.000317,3,2


,indicador,valor
0,Pacientes únicos,1362
1,Ramas Optuna,3
2,Folds por trial,3
3,Trials completos por rama,18
4,Entrenamientos CV de búsqueda objetivo,162
5,Folds de evaluación final,5


In [4]:
spaces = json.loads(
    (RUN_DIR / "config" / "refinement_spaces.json").read_text(encoding="utf-8")
)["spaces"]
space_rows = []
for source_id, space in spaces.items():
    source = next(
        item for item in prepared.references if item["source_candidate_id"] == source_id
    )
    space_rows.append({
        "source_candidate_id": source_id,
        "architecture": source["architecture"],
        "source_learning_rate": source["train_config"]["learning_rate"],
        "node09_learning_rate_min": space["learning_rate"][0],
        "node09_learning_rate_max": space["learning_rate"][1],
        "dropout_range": tuple(space["dropout"]),
        "weight_decay_range": tuple(space["weight_decay"]),
        "tabular_embedding_dim": tuple(space["tabular_embedding_dim"]),
        "feature_top_k": tuple(space["feature_top_k"]),
        "pca_variance": tuple(space["pca_variance"]),
    })
display(pd.DataFrame(space_rows))


,source_candidate_id,architecture,source_learning_rate,node09_learning_rate_min,node09_learning_rate_max,dropout_range,weight_decay_range,tabular_embedding_dim,feature_top_k,pca_variance
0,23f042f62f40d03e,slab2d,0.000317,0.00007,0.000247,"(0.23006770120532094, 0.38006770120532096)","(4.970547987064239e-07, 2.9823287922385433e-05)","(16, 24, 48)","(32, 64, 128)","(0.95, 0.98, none)"
1,3f9bbf24164bd1dc,3d,0.000317,0.00007,0.000247,"(0.22728152879644464, 0.37728152879644467)","(2.8895967463213424e-06, 0.00017337580477928054)","(16, 24, 48)","(32, 64, 128)","(0.95, 0.98, none)"
2,fb93ea204b0ec32f,2.5d,0.000317,0.00007,0.000247,"(0.22728152879644464, 0.37728152879644467)","(2.8895967463213424e-06, 0.00017337580477928054)","(16, 24, 48)","(32, 64, 128)","(0.95, 0.98, none)"


## 3. Tres búsquedas Optuna finas y reanudables

Son **tres estudios**, no tres trials. Por defecto cada estudio exige 18 trials completos y cada trial recorre tres folds. Los primeros ocho trials completos forman el calentamiento del pruner; después puede cortar configuraciones claramente inferiores. Un apagado conserva SQLite, checkpoints por época e historias.


In [5]:
with RunLock(RUN_DIR / "search.lock"):
    search_result = run_search_stage(prepared, EXPERIMENT, device=DEVICE)
display(search_result.summary)
display(Markdown(
    "Abre o vuelve a ejecutar el **nodo 08** y selecciona "
    f"`node09:{RUN_ID}` para comparar estas trayectorias con los nodos 06 y 07."
))


[I 2026-08-31 01:03:04,809] A new study created in RDB with name: node09_slab2d_23f042f62f


[nodo07] fold=0 epoch=1/32 train=0.7314 val=0.6521
[nodo07] fold=0 epoch=2/32 train=0.7028 val=0.6021
[nodo07] fold=0 epoch=3/32 train=0.6516 val=0.5855
[nodo07] fold=0 epoch=4/32 train=0.6392 val=0.5700
[nodo07] fold=0 epoch=5/32 train=0.6239 val=0.5560
[nodo07] fold=0 epoch=6/32 train=0.6078 val=0.5539
[nodo07] fold=0 epoch=7/32 train=0.6092 val=0.5476
[nodo07] fold=0 epoch=8/32 train=0.6103 val=0.5405
[nodo07] fold=0 epoch=9/32 train=0.5960 val=0.5362
[nodo07] fold=0 epoch=10/32 train=0.5927 val=0.5317
[nodo07] fold=0 epoch=11/32 train=0.5763 val=0.5475
[nodo07] fold=0 epoch=12/32 train=0.5834 val=0.5437
[nodo07] fold=0 epoch=13/32 train=0.5959 val=0.5240
[nodo07] fold=0 epoch=14/32 train=0.5458 val=0.5276
[nodo07] fold=0 epoch=15/32 train=0.5748 val=0.5244
[nodo07] fold=0 epoch=16/32 train=0.5568 val=0.5301
[nodo07] fold=0 epoch=17/32 train=0.5372 val=0.5333
[nodo07] fold=0 epoch=18/32 train=0.5495 val=0.5245
[nodo07] fold=0 epoch=19/32 train=0.5609 val=0.5192
[nodo07] fold=0 epoch

[I 2026-08-31 01:31:26,439] Trial 0 finished with value: 0.545080794944553 and parameters: {'dropout': 0.30506770120532095, 'tabular_embedding_dim': 24, 'augmentation_magnitude': 2.1026067125234453, 'augmentation_probability': 0.8009586631176213, 'rotation_degrees': 1.5935397493341092, 'learning_rate': 0.00013134702829321461, 'weight_decay': 3.8501699150849195e-06, 'auxiliary_weight': 0.14550204100106018, 'consistency_weight': 0.0653384855658812, 'feature_top_k': 64, 'pca_variance': 'none'}. Best is trial 0 with value: 0.545080794944553.


[nodo07] fold=0 epoch=1/32 train=0.7362 val=0.6607
[nodo07] fold=0 epoch=2/32 train=0.6945 val=0.5735
[nodo07] fold=0 epoch=3/32 train=0.6397 val=0.5603
[nodo07] fold=0 epoch=4/32 train=0.6301 val=0.5605
[nodo07] fold=0 epoch=5/32 train=0.6250 val=0.5446
[nodo07] fold=0 epoch=6/32 train=0.6248 val=0.5394
[nodo07] fold=0 epoch=7/32 train=0.6290 val=0.5285
[nodo07] fold=0 epoch=8/32 train=0.6040 val=0.5210
[nodo07] fold=0 epoch=9/32 train=0.6051 val=0.5170
[nodo07] fold=0 epoch=10/32 train=0.5953 val=0.5144
[nodo07] fold=0 epoch=11/32 train=0.5832 val=0.5220
[nodo07] fold=0 epoch=12/32 train=0.5979 val=0.5244
[nodo07] fold=0 epoch=13/32 train=0.6044 val=0.5039
[nodo07] fold=0 epoch=14/32 train=0.5585 val=0.5063
[nodo07] fold=0 epoch=15/32 train=0.5852 val=0.5096
[nodo07] fold=0 epoch=16/32 train=0.5557 val=0.5068
[nodo07] fold=0 epoch=17/32 train=0.5340 val=0.5075
[nodo07] fold=0 epoch=18/32 train=0.5426 val=0.5040
[nodo07] fold=0 epoch=19/32 train=0.5516 val=0.5040
[nodo07] fold=0 epoch

[I 2026-08-31 02:04:02,919] Trial 1 finished with value: 0.526610784796341 and parameters: {'dropout': 0.33736718284136136, 'tabular_embedding_dim': 24, 'augmentation_magnitude': 1.8698261938463234, 'augmentation_probability': 0.7935982011508709, 'rotation_degrees': 1.424912319967584, 'learning_rate': 0.00023468160086831781, 'weight_decay': 1.0139295533312985e-05, 'auxiliary_weight': 0.16761014849246753, 'consistency_weight': 0.043237159129940396, 'feature_top_k': 32, 'pca_variance': 'none'}. Best is trial 1 with value: 0.526610784796341.


...

[I 2026-09-01 16:37:40,174] Trial 19 pruned. 


[nodo07] fold=0 epoch=1/32 train=0.7272 val=0.6079
[nodo07] fold=0 epoch=2/32 train=0.6638 val=0.6005
[nodo07] fold=0 epoch=3/32 train=0.6519 val=0.5709
[nodo07] fold=0 epoch=4/32 train=0.6318 val=0.5693
[nodo07] fold=0 epoch=5/32 train=0.6275 val=0.5618
[nodo07] fold=0 epoch=6/32 train=0.6277 val=0.5336
[nodo07] fold=0 epoch=7/32 train=0.6224 val=0.5670
[nodo07] fold=0 epoch=8/32 train=0.6040 val=0.5671
[nodo07] fold=0 epoch=9/32 train=0.6128 val=0.5307
[nodo07] fold=0 epoch=10/32 train=0.6005 val=0.5139
[nodo07] fold=0 epoch=11/32 train=0.5819 val=0.5477
[nodo07] fold=0 epoch=12/32 train=0.5978 val=0.5479
[nodo07] fold=0 epoch=13/32 train=0.5989 val=0.5221
[nodo07] fold=0 epoch=14/32 train=0.5372 val=0.5835
[nodo07] fold=0 epoch=15/32 train=0.5806 val=0.5283
[nodo07] fold=0 epoch=16/32 train=0.5577 val=0.5538
[nodo07] fold=0 epoch=17/32 train=0.5453 val=0.5643
[nodo07] fold=0 epoch=18/32 train=0.5418 val=0.5520
[nodo07] fold=0 epoch=19/32 train=0.5584 val=0.5537
[nodo07] fold=1 epoch

[I 2026-09-01 17:13:21,639] Trial 20 finished with value: 0.5408736737092611 and parameters: {'dropout': 0.2726340961044135, 'tabular_embedding_dim': 24, 'augmentation_magnitude': 2.0417203891792073, 'augmentation_probability': 0.9013329878987006, 'rotation_degrees': 1.1540778411112926, 'learning_rate': 0.0001795213430490994, 'weight_decay': 1.8247973520742044e-05, 'auxiliary_weight': 0.17800919828542902, 'consistency_weight': 0.040752715836708794, 'feature_top_k': 32, 'pca_variance': 'none'}. Best is trial 15 with value: 0.5361371208029735.


,study_name,source_candidate_id,source_node07_rank,selection_roles,architecture,feature_variant,lateral_strategy,completed_trials,total_trials,search_log_loss,fixed_epochs,candidate_id
0,node09_slab2d_23f042f62f,23f042f62f40d03e,1,top_1_overall,slab2d,image_radiomics_sbr,random_flip,18,18,0.524060,25,555b81376733bb67
1,node09_25d_fb93ea204b,fb93ea204b0ec32f,2,top_2_overall,2.5d,image_radiomics_sbr,random_flip,18,20,0.527828,15,c56be99202976455
2,node09_3d_3f9bbf2416,3f9bbf24164bd1dc,3,best_3d,3d,image_radiomics_sbr,random_flip,18,21,0.536137,18,a37f424ba911f6cc


Abre o vuelve a ejecutar el **nodo 08** y selecciona `node09:node09_fine_v1` para comparar estas trayectorias con los nodos 06 y 07.

## 4. Evaluación final de los tres ganadores

Optuna entrega un ganador por cada rama. Sólo esos tres modelos se reentrenan en los cinco folds congelados; son 15 entrenamientos finales. El fold externo no controla early stopping: el número de épocas proviene de la mediana de la búsqueda, con un mínimo de ocho para evitar el entrenamiento demasiado corto observado antes.


In [6]:
with RunLock(RUN_DIR / "final.lock"):
    final_result = run_final_stage(prepared, EXPERIMENT, device=DEVICE)
display(final_result.metrics)
display(Markdown(
    f"**Modelo primario nodo 09:** `{final_result.deployment_manifest['primary_candidate_id']}` · "
    f"**modelos por fold del primario:** `{final_result.deployment_manifest['n_fold_models']}`"
))


[nodo07] fold=0 epoch=1/25 train=0.7412 val=nan
[nodo07] fold=0 epoch=2/25 train=0.7068 val=nan
[nodo07] fold=0 epoch=3/25 train=0.6499 val=nan
[nodo07] fold=0 epoch=4/25 train=0.6299 val=nan
[nodo07] fold=0 epoch=5/25 train=0.6163 val=nan
[nodo07] fold=0 epoch=6/25 train=0.5824 val=nan
[nodo07] fold=0 epoch=7/25 train=0.6003 val=nan
[nodo07] fold=0 epoch=8/25 train=0.5702 val=nan
[nodo07] fold=0 epoch=9/25 train=0.5926 val=nan
[nodo07] fold=0 epoch=10/25 train=0.5957 val=nan
[nodo07] fold=0 epoch=11/25 train=0.5761 val=nan
[nodo07] fold=0 epoch=12/25 train=0.6042 val=nan
[nodo07] fold=0 epoch=13/25 train=0.5551 val=nan
[nodo07] fold=0 epoch=14/25 train=0.5615 val=nan
[nodo07] fold=0 epoch=15/25 train=0.5603 val=nan
[nodo07] fold=0 epoch=16/25 train=0.5593 val=nan
[nodo07] fold=0 epoch=17/25 train=0.5543 val=nan
[nodo07] fold=0 epoch=18/25 train=0.5524 val=nan
[nodo07] fold=0 epoch=19/25 train=0.5560 val=nan
[nodo07] fold=0 epoch=20/25 train=0.5708 val=nan
[nodo07] fold=0 epoch=21/25 t

,candidate_id,architecture,feature_variant,lateral_strategy,raw_log_loss,raw_brier,raw_auc,raw_balanced_accuracy_0_5,raw_ece_10,calibrated_log_loss,calibrated_brier,calibrated_auc,calibrated_balanced_accuracy_0_5,calibrated_ece_10,deployment_temperature
0,555b81376733bb67,slab2d,image_radiomics_sbr,random_flip,0.547940,0.182645,0.801811,0.734904,0.050513,0.543655,0.181365,0.800133,0.734904,0.037625,1.234490
1,c56be99202976455,2.5d,image_radiomics_sbr,random_flip,0.561650,0.188128,0.785729,0.740164,0.044513,0.558443,0.187208,0.784103,0.740164,0.042201,1.210187
2,a37f424ba911f6cc,3d,image_radiomics_sbr,random_flip,0.610864,0.202112,0.775264,0.713962,0.087669,0.571392,0.193613,0.772863,0.713962,0.031732,1.737770


**Modelo primario nodo 09:** `555b81376733bb67` · **modelos por fold del primario:** `5`

## Lectura del resultado

El nodo 09 mejora la resolución de la optimización, pero no garantiza una mejora real. Se promueve un candidato sólo si el CV5 congelado supera al nodo 07 en log loss cross-calibrado y conserva estabilidad por fold y familia. Para revisar progreso parcial, curvas de aprendizaje, pruning, calibración y comparación entre redes, usa el nodo 08.
